# Init Lakehouse

In [1]:
%%configure -f
{
    "defaultLakehouse": {"name": "DE_LH_100_BondedWarehouse"}
}

StatementMeta(, d39a0bfc-6a9f-4848-9d92-fc7475c030b5, -1, Finished, Available, Finished)

# Init Imports (these need cutting-down post creation)

In [2]:
import os
import csv
import re
import shutil
import unicodedata
import pandas as pd

import notebookutils

#from decimal import Decimal
from datetime import datetime
from datetime import timedelta
#from collections import Counter
#from functools import reduce
import time

#from pyspark import StorageLevel
from pyspark.sql import DataFrame, Row
from pyspark.sql.functions import col, lit, when, concat, concat_ws, coalesce, count, monotonically_increasing_id, sum, to_date, udf, current_timestamp, length, substring, split, size, asc, row_number
from pyspark.sql.functions import broadcast, hash, array, expr, array_distinct, date_format
from pyspark.sql.types import *
#from pyspark.sql import Window
from pyspark.sql import functions as F
from delta.tables import DeltaTable

StatementMeta(, d39a0bfc-6a9f-4848-9d92-fc7475c030b5, 3, Finished, Available, Finished)

# Init Export Process

In [3]:
def save_dataframe_to_csv(df, file_path, show_header=False, mode='overwrite'):
    """
    Save a DataFrame as a single CSV file in a PySpark application.

    Parameters:
    df (pyspark.sql.DataFrame): The DataFrame to save.
    file_path (str): The path to save the CSV file.
    header (bool): Whether to include the header in the CSV file. Default is True.
    mode (str): The write mode. Options are 'overwrite', 'append', 'ignore', 'error' or 'errorifexists'. Default is 'overwrite'.

    Returns:
    None
    """

    use_pipes = len(df.columns) != 1
    print(f'Add pipes: {use_pipes}')
    print(f'Show headers: {show_header}')

    pandas_df = df.toPandas()
    
    # Replace newlines and carriage returns
    pandas_df = pandas_df.replace({r'\r\n': ' ', r'\n': ' ', r'\r': ' '}, regex=True)

    # Create a string representation of the DataFrame with '|' as separator
    # Escape special characters such as commas and pipes
    if use_pipes:
        csv_data = pandas_df.to_csv(sep="|", index=False, header=show_header, quoting=csv.QUOTE_NONE, escapechar="\\")
    else:
        csv_data = pandas_df.to_csv(sep="~", index=False, header=show_header, quoting=csv.QUOTE_NONE, escapechar="\\")

    # Add trailing pipe '|' at the end of each line
    csv_data_with_pipe = '\n'.join([line + '|' for line in csv_data.split('\n') if line])

    # Write to the file
    with open(file_path, 'w') as f:
        f.write(csv_data_with_pipe)


StatementMeta(, d39a0bfc-6a9f-4848-9d92-fc7475c030b5, 4, Finished, Available, Finished)

# Init Debug & Incremental Vars

In [4]:
workspace_name = notebookutils.mssparkutils.env.getWorkspaceName()

if "DEV" in workspace_name.upper():
    debug = True
    incremental_run = False
    default_days_lag: int = 7

elif "UAT" in workspace_name.upper():
    debug = True
    incremental_run = True
    default_days_lag: int = 7

else:
    debug = False
    incremental_run = True
    default_days_lag: int = 1

if debug:
    print(debug , incremental_run)

StatementMeta(, d39a0bfc-6a9f-4848-9d92-fc7475c030b5, 5, Finished, Available, Finished)

True False


# Init Days Lag Var

In [5]:
filterdate_pipe = ''

#default_days_lag: int = 1

enable_string_truncation = True
create_hash_cols: bool = False
transfer_file: bool = False
retain_error_records_in_ouput_file: bool = False

# Override Debug

#debug = False   #<<<<<<<<<<<<<<<<<<<<<<<<<<<  <<<<<<<<<<<<<<<<<<<<
#debug = True   #<<<<<<<<<<<<<<<<<<<<<<<<<<<  <<<<<<<<<<<<<<<<<<<<

# Override Full Run 

#incremental_run = False    #<<<<<<<<<<<<<<<<<<<<<<<<<<<  <<<<<<<<<<<<<<<<<<<<
#incremental_run = True     #<<<<<<<<<<<<<<<<<<<<<<<<<<<  <<<<<<<<<<<<<<<<<<<<

StatementMeta(, d39a0bfc-6a9f-4848-9d92-fc7475c030b5, 6, Finished, Available, Finished)

In [6]:
filterdate = datetime.now() - timedelta(days=default_days_lag)
filterdate = filterdate.date()

if debug:
    print(f'Get Closing Stock from: {filterdate}')

StatementMeta(, d39a0bfc-6a9f-4848-9d92-fc7475c030b5, 7, Finished, Available, Finished)

Get Closing Stock from: 2025-05-13


# Init Query(s)

In [7]:
closing_stock_df = spark.sql(f"""
SELECT 
'ENDCTG' AS Company_Code,
'' AS Site_Code,
current_date() - 1 AS Date,
inventsum.itemid AS Product_Part_No_Ref,
CAST(SUM(inventsum.physicalinvent) as Decimal(10, 2)) AS Quantity

FROM inventsum

INNER JOIN inventtable 
ON inventsum.itemid = inventtable.itemid
AND inventsum.dataareaid = inventtable.dataareaid

INNER JOIN ecoresproduct
ON inventtable.product = ecoresproduct.recid

WHERE inventsum.dataareaid IN ('end.','END.')
AND inventsum.inventlocationid = 'PAR'
AND inventsum.wmslocationid != 'Off-site'
--AND inventsum.physicalinvent > 0

GROUP BY
     current_date()
    ,inventsum.itemid

"""
)
if debug:
      display(closing_stock_df)

StatementMeta(, d39a0bfc-6a9f-4848-9d92-fc7475c030b5, 8, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 857dfdaf-d917-4487-8a8a-425816509109)

In [8]:
closing_stock_df = closing_stock_df.select(
    substring(col("Company_Code").cast("string"),1, 10).alias("Company_Code"),
    col("Site_Code").cast("string").alias("Site_Code"),
    col("Date").cast("date").alias("Date"),
    substring(col("Product_Part_No_Ref").cast("string"),1, 7).alias("Product_Part_No_Ref"),
    col("Quantity").cast("string").alias("Quantity")

)
if debug:
    display(closing_stock_df)

StatementMeta(, d39a0bfc-6a9f-4848-9d92-fc7475c030b5, 9, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, c299f796-9710-417f-826c-f386d455a8e3)

# N/A to NA

In [9]:
closing_stock_df = closing_stock_df.replace("N/A", "NA")

StatementMeta(, d39a0bfc-6a9f-4848-9d92-fc7475c030b5, 10, Finished, Available, Finished)

## Date Field Changing Post Query(s)

In [10]:
list_date_columns_1 = [name for name, dtype in closing_stock_df.dtypes if dtype in ('date','timestamp')]

if debug:
    print("Date Columns to change: " , list_date_columns_1)

StatementMeta(, d39a0bfc-6a9f-4848-9d92-fc7475c030b5, 11, Finished, Available, Finished)

Date Columns to change:  ['Date']


In [11]:
for column in list_date_columns_1:
    closing_stock_df = closing_stock_df.withColumn(column, date_format(column, "dd-MM-yyyy"))

StatementMeta(, d39a0bfc-6a9f-4848-9d92-fc7475c030b5, 12, Finished, Available, Finished)

In [12]:
if debug:
    display(closing_stock_df)

StatementMeta(, d39a0bfc-6a9f-4848-9d92-fc7475c030b5, 13, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, b48eebe0-f70c-4513-b3c9-72272e0f7ed7)

# Init Good File Name & Date Logic

In [13]:
file_path_folder = "/lakehouse/default/Files/Output/"
file_extention = '.dat'

# file date time - stamp tomorrow's date if after 6.15pm --Nick: had to knock it back 1 hour to account for timezone difference; working
now = datetime.now()
cutoff_time = now.replace(hour=17, minute=15, second=0, microsecond=0)

now -= timedelta(days=1)
#file_datetime = tomorrow.strftime('%Y-%m-%d')
file_datetime = now.strftime('%Y-%m-%d-%H')


file_name = "closingstock" + "_" + file_datetime + file_extention
file_path = file_path_folder + file_name

StatementMeta(, d39a0bfc-6a9f-4848-9d92-fc7475c030b5, 14, Finished, Available, Finished)

In [14]:
if debug:
    print("Now: " , now)
    print("Cutoff: " , cutoff_time)
    print("File Date: " , file_datetime)
    print("File Folder Path: " , file_path_folder)
    print("Good File Name: " , file_name)
    print("Good File Name: " , file_path)

StatementMeta(, d39a0bfc-6a9f-4848-9d92-fc7475c030b5, 15, Finished, Available, Finished)

Now:  2025-05-19 10:32:29.332300
Cutoff:  2025-05-20 17:15:00
File Date:  2025-05-19-10
File Folder Path:  /lakehouse/default/Files/Output/
Good File Name:  closingstock_2025-05-19-10.dat
Good File Name:  /lakehouse/default/Files/Output/closingstock_2025-05-19-10.dat


# Export Good

In [15]:
save_dataframe_to_csv(closing_stock_df, file_path, show_header=False)

StatementMeta(, d39a0bfc-6a9f-4848-9d92-fc7475c030b5, 16, Finished, Available, Finished)

Add pipes: True
Show headers: False


In [16]:
if closing_stock_df.take(1):

    ready_to_copy = True

else:

    ready_to_copy = False

if debug:
    print(ready_to_copy)

StatementMeta(, d39a0bfc-6a9f-4848-9d92-fc7475c030b5, 17, Finished, Available, Finished)

True


# Init Record Tracking

In [17]:
if incremental_run:

    # Save the final_df to different tables based on the exportfile variable value

    def record_tracking_df_to_table(dataframe, table_name, file_name):
        """Saves distinct records to a table, checking for duplicates and enabling column mapping."""
        table_name_lower = table_name.lower()

        # Check if table exists
        table_exists = True
        try:
            spark.read.table(table_name_lower)
            print(f"Table {table_name_lower} exists.")
        except Exception as e:
            print(f"Table {table_name_lower} does not exist.")
            table_exists = False

        # Columns to deduplicate on (excluding metadata)
        dedup_cols = [col for col in dataframe.columns if col not in ["Timestamp", "ExportName", "ExportDate"]]

        if table_exists:
            try:
                dataframe = dataframe.withColumn("ExportName", lit(file_name).cast(StringType())) \
                                    .withColumn("ExportDate", current_timestamp().cast(TimestampType()))

                existing_df = spark.read.table(table_name_lower)

                distinct_existing_df = existing_df.dropDuplicates(subset=dedup_cols)
                initial_existing_count = distinct_existing_df.count()
                print(f"Existing distinct count: {initial_existing_count}")

                distinct_new_df = dataframe.dropDuplicates(subset=dedup_cols)

                combined_distinct_df = distinct_new_df.unionByName(distinct_existing_df) \
                                                    .dropDuplicates(subset=dedup_cols)
                final_distinct_count = combined_distinct_df.count()
                print(f"Final distinct count: {final_distinct_count}")

                rows_added = final_distinct_count - initial_existing_count
                print(f"Added {rows_added} new distinct records to {table_name_lower}.")

                combined_distinct_df.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name_lower)
                print(f"Distinct records saved to {table_name_lower}.")

            except Exception as e:
                print(f"Exception: Saving all records in new table. Exception: {e}")
                dataframe = dataframe.dropDuplicates(subset=dedup_cols)
                dataframe.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name_lower)
                print(f"Distinct records saved to {table_name_lower}.")

        else:
            try:
                print(f"Table doesn't exist. Saving all records in new table.")
                dataframe = dataframe.withColumn("ExportName", lit(file_name).cast(StringType())) \
                                    .withColumn("ExportDate", current_timestamp().cast(TimestampType()))
                dataframe = dataframe.dropDuplicates(subset=dedup_cols)
                dataframe.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name_lower)
                print(f"Distinct records saved to {table_name_lower}.")
            except Exception as e:
                print(f"Error saving data: {e}")

    table_prefix = 'BondedWarehouseRecordTracking_'

    record_tracking_df_to_table(closing_stock_df, f"{table_prefix}closingstock", file_name)

StatementMeta(, d39a0bfc-6a9f-4848-9d92-fc7475c030b5, 18, Finished, Available, Finished)

# Init Send To Azure Blob Storage

In [18]:
if ready_to_copy == False:
    output_msg = f'Process Complete'

    notebookutils.notebook.exit(output_msg)

StatementMeta(, d39a0bfc-6a9f-4848-9d92-fc7475c030b5, 19, Finished, Available, Finished)

## Copy the file to an ADLS account for loading to the SFTP
Set the source and destination paths

In [19]:
if "DEV" in workspace_name.upper():
    dest_abfss_file_path = "Files/bonded_warehouse_dev/ToBeSent/" + file_name
    
elif "UAT" in workspace_name.upper():
    dest_abfss_file_path = "Files/bonded_warehouse_uat/ToBeSent/" + file_name

else:
    dest_abfss_file_path = "Files/bonded_warehouse/ToBeSent/" + file_name

StatementMeta(, d39a0bfc-6a9f-4848-9d92-fc7475c030b5, 20, Finished, Available, Finished)

In [20]:
source_abfss_file_path = 'Files/Output/' + file_name

StatementMeta(, d39a0bfc-6a9f-4848-9d92-fc7475c030b5, 21, Finished, Available, Finished)

In [21]:
if transfer_file:
    notebookutils.fs.fastcp(source_abfss_file_path, dest_abfss_file_path)

StatementMeta(, d39a0bfc-6a9f-4848-9d92-fc7475c030b5, 22, Finished, Available, Finished)

In [22]:
if ready_to_copy == True:
    output_msg = f'Process Complete'

notebookutils.notebook.exit(output_msg)

StatementMeta(, d39a0bfc-6a9f-4848-9d92-fc7475c030b5, 23, Finished, Available, Finished)

ExitValue: Process Complete